# dala-dock: why vancomycin resistance happens

We dock small peptide **candidates** into the antibiotic **vancomycin** and watch, stage by stage,
why the native **D-Ala-D-Ala** cell-wall terminus binds tightly while the resistant
**D-Ala-D-Lac** variant does not.

Runs anywhere: **Vina** on a laptop (no GPU), **AutoDock-GPU** automatically on a GPU cluster.

**How to use:** run each cell top to bottom. Every stage prints what it did and shows a 3D picture.

**Setup.** Import the pipeline (`prep_receptor`, `box`, `prep_ligand`, `dock`, `analyze`, `viz`), set a `work/` folder, and detect the engine. `dock.detect_engine()` picks **AutoDock-GPU** if a GPU is available, otherwise **Vina** — the rest of the notebook runs the same either way.

In [ ]:
# --- setup: make the daladock package importable and set a work directory ---
%load_ext autoreload
%autoreload 2
import sys, os, pathlib
# drop any PYTHONPATH-injected dirs that could shadow this env's packages
for _p in os.environ.get('PYTHONPATH', '').split(os.pathsep):
    while _p and _p in sys.path:
        sys.path.remove(_p)

# find the repo root robustly - walk up until we see src/daladock.
# (works whether the cwd is notebooks/, work/, or the repo root, and is safe to re-run)
_here = pathlib.Path.cwd()
REPO = next((c for c in [_here, *_here.parents] if (c / 'src' / 'daladock').is_dir()), _here)
sys.path.insert(0, str(REPO / 'src'))
WORK = REPO / 'work'; WORK.mkdir(exist_ok=True); os.chdir(WORK)
DATA = REPO / 'data'

import pandas as pd
import matplotlib.pyplot as plt
from daladock import prep_receptor, box, prep_ligand, dock, analyze, viz
print('repo:', REPO)
print('working dir:', WORK)
print('docking engine that will be used:', dock.detect_engine())

## Stage 1 - Prepare the receptor (vancomycin)

Fetch **1FVM** (vancomycin + D-Ala-D-Ala) and keep vancomycin (chain A). It's a glycopeptide with
non-standard residues, so `prepare_receptor` auto-selects **OpenBabel**.

**What's a PDBQT?** Docking can't read a plain `.pdb` — it needs a **PDBQT = PDB + Q + T**:
- **PDB** — each atom's element and 3-D coordinates (x, y, z)
- **Q** — a **partial charge** per atom (Gasteiger), used for the electrostatics term
- **T** — an **AutoDock atom type** (e.g. `OA` = H-bond-accepting O, `HD` = polar H, `A` = aromatic C)

The receptor is **rigid** (no rotatable bonds).

In [ ]:
rec = prep_receptor.prepare_receptor('1FVM', out='vancomycin', keep_chains=['A'],
                                     extract_ref=False)
viz.view_structure(rec['pdbqt'])   # vancomycin in 3D

## Stage 2 - Define the search box

AutoDock never finds pockets on its own — you must give it a **box** to search inside. We place that
box over vancomycin's binding cleft using the position of the D-Ala-D-Ala bound in the 1FVM crystal
(`data/dala_ref.pdb`): its coordinates mark exactly where the site is (the reference ligand is only
used to *locate* the box, not docked). Alternatives: `fpocket` to auto-detect a pocket (verify it!) or
`blind` to search the whole target.

**The numbers:** `define_box` returns `{center:[x,y,z], size:[sx,sy,sz]}` in Ångström. With
`detect='ref'`, the **center** is the average position of the reference ligand's atoms and the
**size** is their extent + 8 Å padding. The magenta box is the region AutoDock will search.

In [ ]:
b = box.define_box(rec['pdbqt'], ref=str(DATA / 'dala_ref.pdb'), detect='ref')  #ref/fpocket/blind
viz.view_box(rec['pdbqt'], b['center'], b['size'])

## Stage 3 - Prepare the candidates

The panel in `data/candidates.csv`: native terminus, the two real resistance variants
(D-Ala-D-Lac, D-Ala-D-Ser), and controls.

**Ligand PDBQT — the extra piece.** Each SMILES → 3-D coordinates (RDKit) → PDBQT. Unlike the
receptor, a *ligand* PDBQT also stores a **torsion tree** (`ROOT` / `BRANCH` / `TORSDOF`) marking
**rotatable bonds**, so docking can flex the molecule. `TORSDOF` (torsional degrees of freedom) feeds
the entropy penalty in the score below.

In [ ]:
cands = pd.read_csv(DATA / 'candidates.csv')
display(cands)
ligands = {}
for _, row in cands.iterrows():
    ligands[row['name']] = prep_ligand.prepare_ligand(row['smiles'], out=row['name'])
viz.view_ligand(ligands['D-Ala-D-Ala']['pdbqt'])   # peek at one candidate

## Stage 4 - Dock every candidate

`engine='auto'` uses Vina on a laptop or AutoDock-GPU on a GPU node. Each run produces several
**poses** (tries) - we keep them all so we can inspect them next.

### How docking scores a pose

Each pose gets a predicted **binding free energy ΔG (kcal/mol)** — *more negative = tighter*. It's an
**empirical sum of pairwise atom–atom terms**, evaluated over ligand–receptor atom pairs as a function
of distance:

`ΔG ≈ (van der Waals) + (hydrogen bonds) + (electrostatics) + (hydrophobic) − (desolvation) − (entropy penalty × rotatable bonds)`

- **van der Waals** — shape / close-packing complementarity (attraction, then repulsion if too close)
- **hydrogen bonds & electrostatics** — favorable N/O contacts and charge interactions (uses the **Q** in the PDBQT)
- **desolvation** — the cost of pushing water out of the pocket
- **entropy penalty** — each rotatable bond the ligand freezes on binding costs free energy (uses **TORSDOF**)

**Computed fast via grid maps.** Rather than recompute the receptor's field for every pose, `autogrid`
precomputes it once over the box: at each grid point it stores the energy a **probe atom** would feel
from the whole receptor — **one map per ligand atom type** (`C.map`, `OA.map`, `HD.map`, …), plus an
**electrostatic** and a **desolvation** map. A ligand atom's receptor-interaction energy is just the
map value at its position (interpolated), so it changes as the atom moves. Ligand total:

`E_inter = Σ_atoms [ affinity_map(type, xyz) + q · e_map(xyz) + desolv(xyz) ]`

**Binding energy is a *change*, not one molecule minus another:**

`ΔG_bind = E(complex) − E(target) − E(candidate)`

— the bound complex relative to the *separated* receptor and ligand. Vina and AutoDock-GPU use
different formulas (so absolute numbers differ) — trust the **ranking**, not the exact kcal/mol.

In [ ]:
results = {}
for name, lig in ligands.items():
    print(name)
    results[name] = dock.dock(rec['pdbqt'], lig['pdbqt'], b['center'], b['size'],
                              engine='auto', out=name, n_poses=10)

## Stage 4b - Look inside one candidate's poses

Docking returns many **poses** (attempts), each with an energy. Let's open up native **D-Ala-D-Ala**.
Per pose:
- **affinity** - predicted binding energy (more negative = better)
- **rmsd_to_best** - how far this pose sits from the best pose (Angstrom)
- **cluster / cluster_size** - poses within ~2 A are grouped; a big low-energy cluster = confident answer.

In [ ]:
candidate = 'D-Ala-D-Ala'
r = results[candidate]
poses_tbl = analyze.cluster_poses(r['poses'], r['scores'])
print(f"{candidate}: {len(r['scores'])} poses, engine = {r['engine']}")
display(poses_tbl)

**Pose energies at a glance.** Each bar is one pose's ΔG; **red = the best cluster**. A clear low-energy group (vs scattered bars) is a good sign.

In [ ]:
plt.figure(figsize=(6, 3))
colors = ['crimson' if c == 1 else 'steelblue' for c in poses_tbl['cluster']]
plt.bar(poses_tbl['pose'], poses_tbl['affinity_kcal_mol'], color=colors)
plt.xlabel('pose #'); plt.ylabel('affinity (kcal/mol)')
plt.title(f'{candidate}: pose energies  (red = best cluster)')
plt.tight_layout(); plt.show()

### All 10 poses overlaid
Each pose a different color. Good poses sit in the cleft; bad (high-energy) ones drift into solvent.

In [ ]:
viz.view_poses(rec['pdbqt'], r['poses'], max_poses=10)

### Best pose vs. a bad pose
First = pose 1 (lowest energy, in the pocket). Second = worst-scoring pose (drifted away).

In [ ]:
best_pose = int(poses_tbl.iloc[0]['pose'])
viz.view_pose_n(rec['pdbqt'], r['poses'], best_pose,
                label=f"pose {best_pose}: {r['scores'][best_pose-1]:.2f} kcal/mol (best)")

In [ ]:
worst_pose = int(poses_tbl.sort_values('affinity_kcal_mol').iloc[-1]['pose'])
viz.view_pose_n(rec['pdbqt'], r['poses'], worst_pose,
                label=f"pose {worst_pose}: {r['scores'][worst_pose-1]:.2f} kcal/mol (worst)")

### What "binding" actually is — the hydrogen bonds

The candidate is **not chemically bonded** to vancomycin. It's held in the pocket by weak,
reversible **non-covalent** interactions — chiefly **hydrogen bonds**. Below, each dashed yellow
line is a candidate↔vancomycin H-bond contact (electronegative N/O atoms ~2.4–3.5 Å apart), with
the distance labeled. This is what *"bound"* means — close contact and these interactions, **not**
a new covalent bond. (The resistant D-Ala-D-Lac loses one of these H-bonds — that's the whole story.)

In [ ]:
# Dashed yellow lines = candidate<->vancomycin hydrogen bonds (distances in Angstrom).
# These non-covalent contacts are what "binding" means - the two molecules are NOT bonded.
viz.view_hbonds(rec['pdbqt'], results['D-Ala-D-Ala']['poses'], pose=1)

### Step through every pose (interactive)
Drag the slider to view each pose with its energy. (Interactive only in a live kernel.)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def show_pose(pose=1):
    aff = r['scores'][pose - 1]
    display(viz.view_pose_n(rec['pdbqt'], r['poses'], pose,
                            label=f'pose {pose}:  {aff:.2f} kcal/mol'))

widgets.interact(show_pose, pose=widgets.IntSlider(min=1, max=len(r['scores']), value=1));

## Stage 5 - Leaderboard + best pose

Ranked by predicted binding (more negative = tighter). Expect native **D-Ala-D-Ala** on top and
the resistant variants weaker - the molecular basis of vancomycin resistance.

In [ ]:
board = analyze.leaderboard(results, csv='leaderboard.csv')
display(board)
best = board.iloc[0]['ligand']
print('best binder:', best)
viz.view_complex(rec['pdbqt'], results[best]['poses'])   # vancomycin + top pose

## What to notice (the critical-thinking part)

- **Poses vs. answer:** docking tries many poses; the 'answer' is the lowest-energy one, trusted more
  when many poses cluster there (Stage 4b).
- **Direction vs magnitude:** docking gets the *ordering* right (native tighter) but *underestimates*
  the gap. Real vancomycin binds D-Ala-D-Ala ~1000x tighter than D-Ala-D-Lac. Trust trends, not exact kcal/mol.
- **Scores differ by engine:** Vina vs AutoDock-GPU use different scoring - compare rankings within one engine.
- **Validate first:** on a target with a known ligand, redock it and confirm <~2 A before trusting new scores.

### Try your own
Edit `data/candidates.csv` and re-run Stages 3-5. Different target? Give `prepare_receptor()` another PDB id.

## Your turn (exercises)

Use the functions you've already seen. Every candidate is **PubChem-verified** (see the
`cid` / `source` columns in the CSVs).

**Tier 1 — Interpret + validate.** From the Stage 5 leaderboard: which terminus binds tightest?
Does it match the resistance story (native **D-Ala-D-Ala** > resistant **D-Ala-D-Lac** / **D-Ala-D-Ser**)?
How big is the best cluster (Stage 4b)? Then run the cell below — the docked pose (orange) should
overlap the crystal reference (green). Finally, argue: why is *binding* necessary but not *sufficient*
for an antibiotic?

**Tier 2 — Tweak the parameters.** Re-dock with more sampling and compare the result *and the runtime*
— that's the quality-vs-cost trade-off HPC exists to manage.

**Tier 3 — Add & screen candidates.** Screen the ~50 verified molecules in `data/candidates_scan.csv`
and build a leaderboard. Which terminal residues does vancomycin tolerate? (Add your own from
[PubChem](https://pubchem.ncbi.nlm.nih.gov) to `data/candidates.csv` too.)

In [ ]:
# Tier 1 - validate: docked best pose (orange) vs the crystal reference D-Ala-D-Ala (green).
# If they overlap, docking recovered the real binding mode.
viz.view_pose_vs_reference(rec['pdbqt'], results['D-Ala-D-Ala']['poses'],
                           str(DATA / 'dala_ref.pdb'))

In [ ]:
# Tier 2 - more sampling. Works on whichever engine your machine has:
#   Vina (CPU)      -> uses exhaustiveness + n_poses
#   AutoDock-GPU    -> uses nrun   (each run = one pose, then clustered)
# We pass BOTH knobs, so 'engine=auto' does more sampling regardless of resources.
# Baselines from Stage 4: Vina exhaustiveness=8, n_poses=10 ; AutoDock-GPU nrun=50.
import time
t0 = time.time()
r2 = dock.dock(rec['pdbqt'], ligands['D-Ala-D-Ala']['pdbqt'], b['center'], b['size'],
               engine='auto', out='DAA_more',
               n_poses=20, exhaustiveness=16,   # -> Vina
               nrun=200)                          # -> AutoDock-GPU
dt = time.time() - t0
knob = "nrun=200" if r2['engine'] == 'adgpu' else "exhaustiveness=16, n_poses=20"
print(f"engine = {r2['engine']} ({knob});  took {dt:.1f}s;  best = {r2['best_score']:.2f} kcal/mol")
display(analyze.cluster_poses(r2['poses'], r2['scores']))
# Q: did the best pose/cluster change vs the default run? was the extra compute time worth it?

In [ ]:
# Tier 3 - screen the verified candidate library (all PubChem-sourced; see cid/source columns)
scan = pd.read_csv(DATA / 'candidates_scan.csv')
print(f"{len(scan)} verified candidates available; screening the first 12 "
      f"(change .head(12) to scan.iterrows() to screen them all)")
scan_results = {}
for _, row in scan.head(12).iterrows():
    lig = prep_ligand.prepare_ligand(row['smiles'], out='scan_' + str(row['name']), verbose=False)
    scan_results[row['name']] = dock.dock(rec['pdbqt'], lig['pdbqt'], b['center'], b['size'],
                                          engine='auto', out='scan_' + str(row['name']), verbose=False)
display(analyze.leaderboard(scan_results, csv='leaderboard_scan.csv'))
# Q: which terminal residues does vancomycin tolerate? where does D-Ala-D-Lac rank, and why?